In [1]:
import pandas as pd
import numpy as np
import glob
import yaml

import xml.etree.ElementTree as ET

In [3]:
with open("config_dataset.yaml", "r") as stream:
    config_dataset = yaml.safe_load(stream)

metadata_path = config_dataset['metadata_path']
metadata_path

'data/metadata_ff.csv'

In [4]:
metadata = pd.read_csv(metadata_path)
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info,id_pair
0,TCGA-44-2665-01A-01-TS1.c5b2abc1-bff0-431b-890...,77b82c3f-e20c-4e3b-a55c-08af359bbb6e.rna_seq.a...,TCGA-44-2665,TCGA-44-2665-01A,TCGA-44-2665-01A,Primary,TS,0
1,TCGA-44-2665-01A-01-TS1.c5b2abc1-bff0-431b-890...,af649713-fb41-496a-a94b-3691d202c1c8.rna_seq.a...,TCGA-44-2665,TCGA-44-2665-01A,TCGA-44-2665-01A,Primary,TS,1
2,TCGA-97-7937-01A-01-TS1.52bec102-5ea6-4f89-b3c...,d1eba37d-0b00-47f4-a7c8-d2e80ec3e27b.rna_seq.a...,TCGA-97-7937,TCGA-97-7937-01A,TCGA-97-7937-01A,Primary,TS,2
3,TCGA-44-2665-11A-01-TS1.e284642f-9ea2-473f-b1e...,2b4652fb-eef5-427e-b4f0-24164c042e52.rna_seq.a...,TCGA-44-2665,TCGA-44-2665-11A,TCGA-44-2665-11A,Not Applicable,TS,3
4,TCGA-64-5774-01A-01-TS1.1b1c8581-82a8-43a9-a78...,5a335f07-9b07-46cf-8902-503b24f89bfa.rna_seq.a...,TCGA-64-5774,TCGA-64-5774-01A,TCGA-64-5774-01A,Primary,TS,4
...,...,...,...,...,...,...,...,...
550,TCGA-62-8397-01A-01-TS1.faf5296f-faaf-4d36-8f4...,21a80f43-21be-4549-901b-c99083021c30.rna_seq.a...,TCGA-62-8397,TCGA-62-8397-01A,TCGA-62-8397-01A,Primary,TS,550
551,TCGA-99-8033-01A-01-TS1.c9f52107-d2d5-4e7d-b30...,e0dabde8-1dd8-4aa6-9f23-51dc22fc2cae.rna_seq.a...,TCGA-99-8033,TCGA-99-8033-01A,TCGA-99-8033-01A,Primary,TS,551
552,TCGA-78-8655-01A-01-TS1.5a514f0a-6e90-4ddb-962...,bc8b6dee-0f79-492f-85e1-c1a09d6f9680.rna_seq.a...,TCGA-78-8655,TCGA-78-8655-01A,TCGA-78-8655-01A,Primary,TS,552
553,TCGA-97-7554-01A-01-TS1.f7ebd35c-ecac-4d02-876...,aa305f5b-4c1d-4bc7-81d5-b2b999b327af.rna_seq.a...,TCGA-97-7554,TCGA-97-7554-01A,TCGA-97-7554-01A,Primary,TS,553


In [5]:
def load_last_vist_day(sample_id):
    try:
        path =  glob.glob(f'data/*/*/*.{sample_id}.xml')[0]
        tree = ET.parse(path)
        root = tree.getroot()
        
        # Define the variable you're searching for
        variable_name = "vital_status"
        
        # Search for the variable in the XML tree
        for elem in root.iter():
            if "vital_status" in elem.tag:
                status = elem.text
            if "days_to_death" in elem.tag:
                days_to_death = elem.text 
            if "days_to_last_followup" in elem.tag:
                days_to_last_followup = elem.text
    
        if status == "Alive":
            return (sample_id, status, days_to_last_followup)
        else:
            return (sample_id, status, days_to_death)
    except:
        return (sample_id, np.nan, np.nan)

In [6]:
survival_metadata = metadata.case_id.apply(lambda x: load_last_vist_day(x))
survival_metadata = pd.DataFrame([(r) for r in survival_metadata.values], columns=["case_id", "censored", "event_time"])
survival_metadata

,case_id,censored,event_time
0,TCGA-44-2665,Alive,1301
1,TCGA-44-2665,Alive,1301
2,TCGA-97-7937,Alive,564
3,TCGA-44-2665,Alive,1301
4,TCGA-64-5774,Alive,2676
...,...,...,...
550,TCGA-62-8397,Alive,1289
551,TCGA-99-8033,Dead,656
552,TCGA-78-8655,None,None
553,TCGA-97-7554,Alive,775


In [7]:
survival_metadata = survival_metadata.drop_duplicates('case_id')
survival_metadata = survival_metadata[~survival_metadata.event_time.isna()]
survival_metadata = survival_metadata[survival_metadata.event_time.astype(int) > 0]
survival_metadata

,case_id,censored,event_time
0,TCGA-44-2665,Alive,1301
2,TCGA-97-7937,Alive,564
4,TCGA-64-5774,Alive,2676
5,TCGA-78-7220,Dead,807
6,TCGA-78-7155,Dead,1171
...,...,...,...
549,TCGA-MP-A4TJ,Dead,339
550,TCGA-62-8397,Alive,1289
551,TCGA-99-8033,Dead,656
553,TCGA-97-7554,Alive,775


In [9]:
survival_metadata.sort_values("case_id")

,case_id,censored,event_time
501,TCGA-05-4249,Alive,1523
483,TCGA-05-4250,Dead,121
232,TCGA-05-4389,Alive,1369
466,TCGA-05-4390,Alive,1126
512,TCGA-05-4397,Dead,731
...,...,...,...
543,TCGA-NJ-A55O,Alive,13
530,TCGA-NJ-A55R,Alive,603
307,TCGA-NJ-A7XG,Alive,617
99,TCGA-O1-A52J,Dead,1798


In [ ]:
survival_metadata.censored.value_counts()

censored
Alive    260
Dead     151
Name: count, dtype: int64

In [ ]:
survival_metadata.to_csv(f'{metadata_path.replace(".csv", "")}_survival.csv', index=False)